#  Student Performance Prediction  CRISP-DM Implementation
### Reimplementation of: *"Assessment and Evaluation of Different Machine Learning Algorithms for Predicting Student Performance"*
**Alsariera et al., Computational Intelligence and Neuroscience, 2022**

---
**Dataset:** UCI Student Performance Dataset (Cortez & Silva, 2008)  
**Framework:** CRISP-DM (Cross-Industry Standard Process for Data Mining)  
**Methodology:** Supervised Classification using DT, ANN, SVM, KNN, NB, LinR (+ Bonus: RF, XGBoost)


---
## Phase 1: Business Understanding

### 1.1 Problem Statement
Student academic performance is a critical indicator of educational quality and institutional success. Early prediction of whether a student will pass, fail, or excel allows educators to intervene proactively, allocate resources efficiently, and personalise teaching strategies.

### 1.2 Objective
The paper investigates and compares six mainstream Machine Learning (ML) algorithms for predicting student academic performance. Through a systematic review of 39 studies (20152021), the authors identify:
- Which ML algorithms perform best for student performance prediction
- Which features (demographic, academic, family/personal) are most predictive

### 1.3 Domain Importance
- **For students:** Early identification of at-risk students enables timely support
- **For educators:** Data-driven insights improve teaching strategies
- **For institutions:** Better academic outcomes enhance university rankings and reputation

### 1.4 CRISP-DM Alignment with the Paper
The paper reviews six ML classifiers: Decision Tree (DT), Artificial Neural Network (ANN), Support Vector Machine (SVM), K-Nearest Neighbor (KNN), Naive Bayes (NB), and Linear Regression (LinR). We reimplement these classifiers on the UCI Student Performance dataset and compare results.

### 1.5 Success Criteria
- Reproduce classification accuracy in the range reported by the paper for each algorithm
- Identify the best-performing algorithm on the student dataset
- Paper's reported average accuracies: ANN (85.9%), DT (85%), SVM (83.4%), NB (83%), KNN (80.7%), LinR (55.5%)

---
## Phase 2: Data Understanding

### 2.1 Dataset Overview
The **UCI Student Performance Dataset** (Cortez & Silva, 2008) contains data from two Portuguese secondary schools for two courses:
- **student-mat.csv**  Math course (395 students, 33 features)
- **student-por.csv**  Portuguese language course (649 students, 33 features)

Features span 7 categories: demographic, academic, internal assessment, family/personal, behavioral, communication, and psychological  matching the paper's attribute taxonomy.

In [ ]:
# 
# IMPORTS & CONFIGURATION
# 
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, f1_score,
                             precision_score, recall_score)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance

# Aesthetic settings
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE = ['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0','#00BCD4','#FF5722','#607D8B']
plt.rcParams.update({'figure.dpi': 100, 'figure.facecolor': 'white'})



In [ ]:
# 
# 2.2 LOAD DATASETS
# 
mat = pd.read_csv('student-mat.csv', sep=';')
por = pd.read_csv('student-por.csv', sep=';')

print(f'Math dataset:       {mat.shape[0]} students × {mat.shape[1]} features')
print(f'Portuguese dataset: {por.shape[0]} students × {por.shape[1]} features')
print(f'\nCombined (merged):  {mat.shape[0] + por.shape[0]} records (before deduplication)')

In [ ]:
# 
# 2.3 BASIC DATA INSPECTION
# 
print('=== MATH DATASET  First 5 rows ===')
mat.head()

In [ ]:
print('=== DATA TYPES & MISSING VALUES ===')
info_df = pd.DataFrame({
    'Dtype': mat.dtypes,
    'Non-Null Count': mat.notnull().sum(),
    'Missing': mat.isnull().sum(),
    'Unique Values': mat.nunique()
})
print(info_df.to_string())

In [ ]:
print('=== DESCRIPTIVE STATISTICS (Numeric) ===')
mat.describe().round(2)

In [ ]:
# 
# 2.4 EDA  GRADE DISTRIBUTION (G3 target)
# 
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Grade Distribution (G3  Final Grade)', fontsize=14, fontweight='bold')

for ax, (df, title) in zip(axes, [(mat, 'Math'), (por, 'Portuguese'), 
                                   (pd.concat([mat, por], ignore_index=True), 'Combined')]):
    ax.hist(df['G3'], bins=21, color=PALETTE[0], edgecolor='white', alpha=0.85)
    ax.set_title(f'{title} (n={len(df)})')
    ax.set_xlabel('Final Grade (G3)')
    ax.set_ylabel('Count')
    ax.axvline(df['G3'].mean(), color='red', linestyle='--', label=f'Mean={df["G3"].mean():.1f}')
    ax.legend()

plt.tight_layout()
plt.savefig('fig_grade_distribution.png', bbox_inches='tight')
plt.show()
print(' Note: Grade 0 likely indicates dropouts (absent for final exam)')

In [ ]:
# 
# 2.5 EDA  CATEGORICAL FEATURES OVERVIEW
# 
cat_cols = mat.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

n_cols = 4
n_rows = (len(cat_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows*3.5))
fig.suptitle('Categorical Feature Distributions (Math Dataset)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    vc = mat[col].value_counts()
    axes[i].bar(vc.index, vc.values, color=PALETTE[i % len(PALETTE)], edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=20)

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('fig_categorical_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# 
# 2.6 EDA  CORRELATION HEATMAP (Numeric Features)
# 
num_cols = mat.select_dtypes(include=np.number).columns.tolist()
corr = mat[num_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Matrix  Numeric Features (Math Dataset)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Top correlations with G3
g3_corr = corr['G3'].drop('G3').sort_values(key=abs, ascending=False)
print('\nTop correlations with G3 (Final Grade):')
print(g3_corr.head(10).to_string())

In [ ]:
# 
# 2.7 EDA  KEY FEATURE RELATIONSHIPS
# 
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Key Feature Relationships with Final Grade (G3)', fontsize=14, fontweight='bold')

# Failures vs G3
mat.groupby('failures')['G3'].mean().plot(kind='bar', ax=axes[0,0], color=PALETTE[3], edgecolor='white')
axes[0,0].set_title('Avg G3 by Number of Failures')
axes[0,0].set_xlabel('Past Failures')
axes[0,0].set_ylabel('Avg Final Grade')

# Study time vs G3
study_labels = {1:'<2h', 2:'2-5h', 3:'5-10h', 4:'>10h'}
mat['studytime_label'] = mat['studytime'].map(study_labels)
mat.groupby('studytime')['G3'].mean().plot(kind='bar', ax=axes[0,1], color=PALETTE[1], edgecolor='white')
axes[0,1].set_xticklabels(['<2h','2-5h','5-10h','>10h'], rotation=0)
axes[0,1].set_title('Avg G3 by Study Time')
axes[0,1].set_xlabel('Weekly Study Time')

# Sex vs G3
mat.groupby('sex')['G3'].mean().plot(kind='bar', ax=axes[0,2], color=PALETTE[4], edgecolor='white')
axes[0,2].set_title('Avg G3 by Gender')
axes[0,2].set_xlabel('Gender')
axes[0,2].set_xticklabels(['Female','Male'], rotation=0)

# Higher education aspiration vs G3
mat.groupby('higher')['G3'].mean().plot(kind='bar', ax=axes[1,0], color=PALETTE[2], edgecolor='white')
axes[1,0].set_title('Avg G3 by Higher Education Goal')
axes[1,0].set_xlabel('Wants Higher Education')
axes[1,0].set_xticklabels(['No','Yes'], rotation=0)

# Mother education vs G3
medu_labels = {0:'None',1:'Primary',2:'Middle',3:'Secondary',4:'Higher'}
mat['Medu_label'] = mat['Medu'].map(medu_labels)
mat.groupby('Medu')['G3'].mean().plot(kind='bar', ax=axes[1,1], color=PALETTE[5], edgecolor='white')
axes[1,1].set_xticklabels(['None','Primary','Middle','Secondary','Higher'], rotation=15)
axes[1,1].set_title('Avg G3 by Mother Education')

# Absences vs G3 scatter
axes[1,2].scatter(mat['absences'], mat['G3'], alpha=0.4, color=PALETTE[0], edgecolors='white', linewidth=0.3)
m, b = np.polyfit(mat['absences'], mat['G3'], 1)
axes[1,2].plot(sorted(mat['absences']), [m*x+b for x in sorted(mat['absences'])], color='red', linewidth=1.5)
axes[1,2].set_title(f'G3 vs Absences (r={mat[["absences","G3"]].corr().iloc[0,1]:.2f})')
axes[1,2].set_xlabel('Absences')
axes[1,2].set_ylabel('G3')

plt.tight_layout()
plt.savefig('fig_feature_relationships.png', bbox_inches='tight')
plt.show()

In [ ]:
# 
# 2.8 EDA  GRADE PROGRESSION (G1  G2  G3)
# 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Grade Progression G1  G2  G3', fontsize=13, fontweight='bold')

# Box plots
mat[['G1','G2','G3']].plot(kind='box', ax=axes[0], color=dict(boxes=PALETTE[0], whiskers=PALETTE[1],
                                                                medians='red', caps=PALETTE[2]))
axes[0].set_title('Grade Distribution across Periods')
axes[0].set_ylabel('Grade (020)')

# Correlation scatter G2 vs G3
axes[1].scatter(mat['G2'], mat['G3'], alpha=0.5, color=PALETTE[0], edgecolors='white', linewidth=0.3)
axes[1].set_xlabel('G2 (Second Period)')
axes[1].set_ylabel('G3 (Final)')
r = mat[['G2','G3']].corr().iloc[0,1]
axes[1].set_title(f'G2 vs G3 Correlation (r = {r:.3f})')

plt.tight_layout()
plt.savefig('fig_grade_progression.png', bbox_inches='tight')
plt.show()

print('Key observations:')
print(f'  G1G3 correlation: {mat[["G1","G3"]].corr().iloc[0,1]:.3f}')
print(f'  G2G3 correlation: {mat[["G2","G3"]].corr().iloc[0,1]:.3f}')

---
## Phase 3: Data Preparation

### 3.1 Preprocessing Strategy
Following the paper's methodology, we apply:
1. **Label encoding** for binary/ordinal categorical features
2. **One-hot encoding** for nominal features (school, Mjob, Fjob, reason, guardian)
3. **Target engineering**: Convert G3 (020 numeric) to binary classification: **Pass (G3  10) / Fail (G3 < 10)**
4. **Feature scaling** (StandardScaler) for distance/gradient-based models (SVM, KNN, ANN)
5. **No imputation needed**  dataset has zero missing values

### 3.2 Feature Selection Rationale
Aligned with the paper's identified key attributes:
- **Demographic**: sex, age, address
- **Academic**: G1, G2, failures, absences
- **Family/personal**: Medu, Fedu, Mjob, Fjob, famsize, Pstatus, famsup, paid
- **Behavioral**: studytime, freetime, goout, Dalc, Walc, activities
- **Psychological/social**: romantic, higher, internet

In [ ]:
# 
# 3.1 PREPROCESSING FUNCTION
# 
def preprocess(df, target_col='G3', threshold=10, drop_g1g2=False):
    """
    Preprocess the student dataset:
    - Create binary target: pass/fail
    - Encode categoricals
    - Return X, y
    """
    df = df.copy()
    # Remove helper columns added during EDA
    for c in ['studytime_label','Medu_label']:
        if c in df.columns:
            df = df.drop(columns=[c])
    
    # Binary target
    df['pass'] = (df[target_col] >= threshold).astype(int)
    
    # Columns to drop
    drop_set = {target_col, 'pass'}
    if drop_g1g2:
        drop_set.update(['G1','G2'])
    feature_cols = [c for c in df.columns if c not in drop_set]
    
    # Binary value map
    binary_map = {'yes':1,'no':0,'F':0,'M':1,'U':1,'R':0,'LE3':0,'GT3':1,'T':1,'A':0}
    
    X = df[feature_cols].copy()
    obj_cols = X.select_dtypes(include='object').columns.tolist()
    
    for col in obj_cols:
        unique_vals = set(X[col].dropna().unique())
        if unique_vals <= set(binary_map.keys()):
            X[col] = X[col].map(binary_map)
        else:
            dummies = pd.get_dummies(X[col], prefix=col, drop_first=True)
            X = pd.concat([X.drop(columns=[col]), dummies], axis=1)
    
    y = df['pass']
    return X, y


# Apply preprocessing
X_mat, y_mat = preprocess(mat)
X_por, y_por = preprocess(por)

print(f'Math  Features: {X_mat.shape[1]}, Samples: {X_mat.shape[0]}')
print(f'Pass rate (Math):       {y_mat.mean()*100:.1f}%')
print(f'Portuguese  Features:  {X_por.shape[1]}, Samples: {X_por.shape[0]}')
print(f'Pass rate (Portuguese): {y_por.mean()*100:.1f}%')
print(f'Feature names ({X_mat.shape[1]}):')
print(X_mat.columns.tolist())

In [ ]:
# 
# 3.2 CLASS BALANCE CHECK
# 
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Target Class Distribution (Pass vs Fail)', fontsize=13, fontweight='bold')

for ax, (y, title) in zip(axes, [(y_mat, 'Math'), (y_por, 'Portuguese')]):
    counts = y.value_counts().sort_index()
    bars = ax.bar(['Fail (G3<10)', 'Pass (G310)'], counts.values,
                  color=[PALETTE[3], PALETTE[1]], edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{val}\n({val/len(y)*100:.0f}%)', ha='center', fontsize=11, fontweight='bold')
    ax.set_title(f'{title} Dataset')
    ax.set_ylim(0, max(counts.values) * 1.2)

plt.tight_layout()
plt.savefig('fig_class_balance.png', bbox_inches='tight')
plt.show()

In [ ]:
# 
# 3.3 TRAIN-TEST SPLIT & SCALING
# 
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Math dataset splits
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mat, y_mat, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_mat)

# Portuguese dataset splits
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_por, y_por, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_por)

# Feature scaling (fit on train, transform both)
scaler_m = StandardScaler()
X_train_m_scaled = scaler_m.fit_transform(X_train_m)
X_test_m_scaled  = scaler_m.transform(X_test_m)

scaler_p = StandardScaler()
X_train_p_scaled = scaler_p.fit_transform(X_train_p)
X_test_p_scaled  = scaler_p.transform(X_test_p)

print(f'Math    Train: {len(X_train_m)}, Test: {len(X_test_m)}')
print(f'Por     Train: {len(X_train_p)}, Test: {len(X_test_p)}')
print(' Scaling applied for ANN, SVM, KNN (StandardScaler)')

---
## Phase 4: Modeling

We implement all **six ML algorithms** from the paper:
1. **Decision Tree (DT)**  Paper avg: 85%
2. **Artificial Neural Network (ANN / MLP)**  Paper avg: 85.9% (best)
3. **Support Vector Machine (SVM)**  Paper avg: 83.4%
4. **K-Nearest Neighbor (KNN)**  Paper avg: 80.7%
5. **Naive Bayes (NB)**  Paper avg: 83%
6. **Linear Regression  Logistic Regression (LinR)**  Paper avg: 55.5%

**Bonus models** (not in the paper):
- **Random Forest (RF)**
- **Gradient Boosting (XGB-style)**

In [ ]:
# 
# 4.1 MODEL OVERVIEW
# 
SCALED_MODELS = {'ANN (MLP)', 'SVM', 'KNN', 'Logistic Regression'}

print(f'Total models to evaluate: 8')
paper_models_list = ['Decision Tree','ANN (MLP)','SVM','KNN','Naive Bayes','Logistic Regression']
bonus_models_list = ['Random Forest','Gradient Boosting']
for name in paper_models_list:
    print(f'   Paper  {name}')
for name in bonus_models_list:
    print(f'   Bonus  {name}')

In [ ]:
# 
# 4.2 TRAINING & EVALUATION FUNCTION
# 
def get_fresh_models():
    """Return fresh (unfitted) model instances."""
    return {
        'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
        'ANN (MLP)':           MLPClassifier(hidden_layer_sizes=(128,64), max_iter=500,
                                             activation='relu', solver='adam',
                                             random_state=RANDOM_STATE, early_stopping=True),
        'SVM':                 SVC(kernel='rbf', C=1.0, probability=True, random_state=RANDOM_STATE),
        'KNN':                 KNeighborsClassifier(n_neighbors=5),
        'Naive Bayes':         GaussianNB(),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE),
        'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=RANDOM_STATE),
    }


def evaluate_models(X_train_raw, X_test_raw, y_train, y_test,
                    X_train_scaled, X_test_scaled, dataset_name='Dataset'):
    results = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    models = get_fresh_models()
    
    print()
    print('='*65)
    print('  RESULTS  ' + dataset_name)
    print('='*65)
    print('{:<22} {:<8} {:<8} {:<8} {:<8} {:<8}'.format('Model','Acc','F1','Prec','Rec','CV-Acc'))
    print('-'*65)
    
    for name, model in models.items():
        Xtr = X_train_scaled if name in SCALED_MODELS else X_train_raw
        Xte = X_test_scaled  if name in SCALED_MODELS else X_test_raw
        Xcv = X_train_scaled if name in SCALED_MODELS else X_train_raw
        
        model.fit(Xtr, y_train)
        y_pred = model.predict(Xte)
        
        acc  = accuracy_score(y_test, y_pred)
        f1   = f1_score(y_test, y_pred, average='weighted')
        prec = precision_score(y_test, y_pred, average='weighted')
        rec  = recall_score(y_test, y_pred, average='weighted')
        cv_scores = cross_val_score(model, Xcv, y_train, cv=cv, scoring='accuracy')
        
        results.append({
            'Model': name,
            'Accuracy': round(acc*100, 2),
            'F1-Score': round(f1*100, 2),
            'Precision': round(prec*100, 2),
            'Recall': round(rec*100, 2),
            'CV-Accuracy': round(cv_scores.mean()*100, 2),
            'CV-Std': round(cv_scores.std()*100, 2),
            'Trained Model': model,
            'y_pred': y_pred
        })
        
        print('{:<22} {:<8.2f} {:<8.2f} {:<8.2f} {:<8.2f} {:.2f}+/-{:.2f}'.format(
            name, acc*100, f1*100, prec*100, rec*100, cv_scores.mean()*100, cv_scores.std()*100))
    
    return pd.DataFrame(results)


# Run evaluations (fresh models per dataset)
results_mat = evaluate_models(
    X_train_m.values, X_test_m.values, y_train_m, y_test_m,
    X_train_m_scaled, X_test_m_scaled,
    dataset_name='Math Dataset'
)

results_por = evaluate_models(
    X_train_p.values, X_test_p.values, y_train_p, y_test_p,
    X_train_p_scaled, X_test_p_scaled,
    dataset_name='Portuguese Dataset'
)

---
## Phase 5: Evaluation

### 5.1 Results Comparison with Original Paper
We compare our achieved accuracies against the average accuracies reported across 39 studies in the paper.

In [ ]:
# 
# 5.1 COMPARISON TABLE vs PAPER
# 
paper_avgs = {
    'Decision Tree': 85.0,
    'ANN (MLP)': 85.9,
    'SVM': 83.4,
    'KNN': 80.7,
    'Naive Bayes': 83.0,
    'Logistic Regression': 55.5,
}

comparison = results_mat[['Model','Accuracy','F1-Score','CV-Accuracy','CV-Std']].copy()
comparison['Paper Avg (%)'] = comparison['Model'].map(paper_avgs)
comparison['Por Accuracy'] = results_por['Accuracy'].values
comparison['Avg (Mat+Por)'] = ((comparison['Accuracy'] + comparison['Por Accuracy']) / 2).round(2)

print('\n FULL COMPARISON: Our Results vs Paper Reported Averages')
print('='*80)
display_cols = ['Model','Accuracy','Por Accuracy','Avg (Mat+Por)','CV-Accuracy','CV-Std','Paper Avg (%)']
print(comparison[display_cols].to_string(index=False))

print('\n  Note: Paper avg is across 39 diverse studies/datasets  direct comparison is approximate.')

In [ ]:
# 
# 5.2 ACCURACY COMPARISON CHART
# 
paper_models = [m for m in comparison['Model'] if m in paper_avgs]
bonus_models = [m for m in comparison['Model'] if m not in paper_avgs]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Accuracy Comparison', fontsize=14, fontweight='bold')

# Math results vs paper
ax = axes[0]
x = np.arange(len(paper_models))
w = 0.3
sub = comparison[comparison['Model'].isin(paper_models)]
ax.bar(x - w, sub['Accuracy'], w, label='Our (Math)', color=PALETTE[0], edgecolor='white')
ax.bar(x, sub['Por Accuracy'], w, label='Our (Portuguese)', color=PALETTE[1], edgecolor='white')
ax.bar(x + w, sub['Paper Avg (%)'], w, label='Paper Avg', color=PALETTE[2], edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels([m.replace(' ',  '\n') for m in paper_models], fontsize=9)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Paper Models: Our Results vs Reported Averages')
ax.legend()
ax.set_ylim(40, 100)

# All models  Math
ax2 = axes[1]
all_models = comparison['Model'].tolist()
colors_bar = [PALETTE[2] if m in bonus_models else PALETTE[0] for m in all_models]
bars = ax2.barh(all_models, comparison['Accuracy'], color=colors_bar, edgecolor='white')
for bar, val in zip(bars, comparison['Accuracy']):
    ax2.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
ax2.set_xlabel('Accuracy (%)')
ax2.set_title('All Models  Math Dataset (Blue=Paper, Orange=Bonus)')
ax2.set_xlim(40, 105)
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('fig_accuracy_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# 
# 5.3 CONFUSION MATRICES  All Paper Models (Math)
# 
paper_model_names = [m for m in results_mat['Model'] if m in paper_avgs]
n_cols = 3
n_rows = (len(paper_model_names) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
fig.suptitle('Confusion Matrices  Paper Models (Math Dataset)', fontsize=13, fontweight='bold')
axes = axes.flatten()

for i, row in results_mat[results_mat['Model'].isin(paper_model_names)].iterrows():
    idx = list(paper_model_names).index(row['Model'])
    cm = confusion_matrix(y_test_m, row['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Fail','Pass'], yticklabels=['Fail','Pass'])
    axes[idx].set_title(f"{row['Model']}\nAcc: {row['Accuracy']}%")
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

for j in range(len(paper_model_names), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('fig_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# 
# 5.4 CLASSIFICATION REPORT  Best Model
# 
best_row = results_mat.loc[results_mat['Accuracy'].idxmax()]
best_name = best_row['Model']
print(f' Best Model (Math): {best_name}  Accuracy: {best_row["Accuracy"]}%')
print()
print(classification_report(y_test_m, best_row['y_pred'], target_names=['Fail','Pass']))

In [ ]:
# 
# 5.5 FEATURE IMPORTANCE  Decision Tree & Random Forest
# 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 15 Feature Importances (Math Dataset)', fontsize=13, fontweight='bold')

for ax, model_name in zip(axes, ['Decision Tree', 'Random Forest']):
    model_row = results_mat[results_mat['Model'] == model_name].iloc[0]
    model = model_row['Trained Model']
    importances = pd.Series(model.feature_importances_, index=X_mat.columns)
    top15 = importances.nlargest(15)
    ax.barh(top15.index[::-1], top15.values[::-1],
            color=PALETTE[0] if model_name=='Decision Tree' else PALETTE[2], edgecolor='white')
    ax.set_title(model_name)
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.savefig('fig_feature_importance.png', bbox_inches='tight')
plt.show()

print('\n Observation: G2, G1 (prior grades) consistently dominate  aligns with paper\'s finding')
print('   that academic attributes (CGPA/grades) are most predictive.')

In [ ]:
# 
# 5.6 MULTI-METRIC RADAR CHART  Paper Models
# 
import matplotlib.patches as mpatches

metrics = ['Accuracy','F1-Score','Precision','Recall','CV-Accuracy']
paper_results = results_mat[results_mat['Model'].isin(paper_avgs.keys())].reset_index(drop=True)
N = len(metrics)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], metrics, size=11)

for i, row in paper_results.iterrows():
    values = [row[m] for m in metrics]
    values += values[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=row['Model'], color=PALETTE[i])
    ax.fill(angles, values, alpha=0.07, color=PALETTE[i])

ax.set_ylim(40, 100)
ax.set_title('Multi-Metric Model Comparison (Math, Paper Models)', pad=20, fontsize=13, fontweight='bold')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.savefig('fig_radar_chart.png', bbox_inches='tight')
plt.show()

In [ ]:
# 
# 5.7 CROSS-VALIDATION BOX PLOT
# 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}
for name, model in get_fresh_models().items():
    Xcv = X_train_m_scaled if name in SCALED_MODELS else X_train_m.values
    scores = cross_val_score(model, Xcv, y_train_m, cv=cv, scoring='accuracy')
    cv_results[name] = scores * 100

fig, ax = plt.subplots(figsize=(12, 5))
cv_df = pd.DataFrame(cv_results)
bp = ax.boxplot([cv_df[col] for col in cv_df.columns],
                labels=cv_df.columns, patch_artist=True,
                medianprops={'color': 'red', 'linewidth': 2})
for patch, color in zip(bp['boxes'], PALETTE[:len(cv_df.columns)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('5-Fold CV Accuracy (%)')
ax.set_title('Cross-Validation Stability (Math Dataset)', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('fig_cv_boxplot.png', bbox_inches='tight')
plt.show()

In [ ]:
# 
# 5.8 WITHOUT G1/G2  Realistic Prediction Scenario
# 
# In real-world early prediction, G1/G2 may not yet be available
X_mat_nograds, y_mat_nograds = preprocess(mat, drop_g1g2=True)
X_tr_ng, X_te_ng, y_tr_ng, y_te_ng = train_test_split(
    X_mat_nograds, y_mat_nograds, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_mat_nograds)

sc_ng = StandardScaler()
X_tr_ng_sc = sc_ng.fit_transform(X_tr_ng)
X_te_ng_sc  = sc_ng.transform(X_te_ng)

results_nograds = evaluate_models(
    X_tr_ng.values, X_te_ng.values, y_tr_ng, y_te_ng,
    X_tr_ng_sc, X_te_ng_sc,
    dataset_name='Math (WITHOUT G1/G2  Early Prediction Scenario)'
)

comp_ng = pd.DataFrame({
    'Model': results_mat['Model'],
    'With G1/G2 (%)': results_mat['Accuracy'],
    'Without G1/G2 (%)': results_nograds['Accuracy']
})
comp_ng['Drop (pp)'] = (comp_ng['With G1/G2 (%)'] - comp_ng['Without G1/G2 (%)']).round(2)
print('\n Impact of removing G1/G2 on prediction accuracy:')
print(comp_ng.to_string(index=False))

In [ ]:
# 
# 5.9 FINAL SUMMARY TABLE
# 
print('\n' + '='*80)
print('  FINAL RESULTS SUMMARY  ALL MODELS AND DATASETS')
print('='*80)

summary = pd.DataFrame({
    'Model': results_mat['Model'],
    'In Paper': results_mat['Model'].apply(lambda x: '' if x in paper_avgs else ' Bonus'),
    'Math Acc (%)': results_mat['Accuracy'],
    'Por Acc (%)': results_por['Accuracy'],
    'Math F1': results_mat['F1-Score'],
    'CV Acc (%)': results_mat['CV-Accuracy'],
    'Paper Avg (%)': results_mat['Model'].map(paper_avgs).fillna('')
})
print(summary.to_string(index=False))

best_paper_model = results_mat[results_mat['Model'].isin(paper_avgs)].loc[
    results_mat[results_mat['Model'].isin(paper_avgs)]['Accuracy'].idxmax()]
best_overall_model = results_mat.loc[results_mat['Accuracy'].idxmax()]

print(f'\n Best Paper Model (Math): {best_paper_model["Model"]}  {best_paper_model["Accuracy"]}%')
print(f' Best Overall Model (Math): {best_overall_model["Model"]}  {best_overall_model["Accuracy"]}%')

---
## Phase 6: Deployment / Practical Reflection

### 6.1 Real-World Deployment Scenario

The best-performing model (Random Forest / ANN) can be deployed in a **Student Early Warning System** within a school management platform:

```
Student Enrollment Data  Feature Extraction  ML Model  Risk Score  Teacher Dashboard
                                                                               Automated Alert
```

**Deployment steps:**
1. Serialize the trained model with `joblib.dump(model, 'student_model.pkl')`
2. Build a REST API endpoint (FastAPI/Flask) accepting student feature JSON
3. Return probability of pass/fail + top contributing features
4. Integrate with school MIS (Management Information System)

### 6.2 Limitations
1. **Dataset size**: 395 (math) / 649 (Portuguese) students is small; generalisation may be limited
2. **Dataset age**: Data from 2 Portuguese schools circa 2006-2008  may not reflect modern students
3. **Class imbalance**: Higher pass rates in Portuguese dataset may bias predictions
4. **G1/G2 dependency**: Top accuracy relies on interim grades (not available at enrollment)
5. **Single institution**: Not generalisable across schools without retraining
6. **Feature gaps**: No online engagement, attendance records, or socioeconomic indices

### 6.3 Ethical Considerations
- **Fairness**: Predictions should not reinforce bias against students from lower-educated families
- **Transparency**: Educators must understand model decisions (favour explainable DT/RF over black-box ANN)
- **Privacy**: Student data must comply with GDPR / FERPA regulations
- **Human oversight**: ML predictions should support, not replace, teacher judgment
- **Self-fulfilling prophecy**: Labelling students as "at-risk" can negatively impact self-esteem

### 6.4 Future Improvements
- Collect more data from multiple institutions and countries
- Add temporal features (grade trends over semesters)
- Explore deep learning (LSTM for sequential grade data)
- Apply SMOTE for class balance
- Implement Explainable AI (SHAP values) for model transparency
- Conduct hyperparameter optimisation (GridSearchCV / Bayesian Optimisation)

In [ ]:
# 
# 6.1 SAVE BEST MODEL FOR DEPLOYMENT
# 
import joblib

best_model_name = results_mat.loc[results_mat['Accuracy'].idxmax(), 'Model']
best_model_obj  = results_mat.loc[results_mat['Accuracy'].idxmax(), 'Trained Model']

joblib.dump(best_model_obj, 'best_student_model.pkl')
joblib.dump(scaler_m, 'scaler_math.pkl')

print(f' Best model "{best_model_name}" saved to best_student_model.pkl')
print(' Scaler saved to scaler_math.pkl')
print()
print('To use for inference:')
print('  model = joblib.load("best_student_model.pkl")')
print('  scaler = joblib.load("scaler_math.pkl")')
print('  X_new = scaler.transform(new_student_features)')
print('  prediction = model.predict(X_new)  # 1=Pass, 0=Fail')
print('  probability = model.predict_proba(X_new)[:,1]  # Pass probability')

In [ ]:
# 
# 6.2 DEMO INFERENCE
# 
print('=== DEMO: Predict for 3 sample students from test set ===')
sample_idx = [0, 5, 10]

for idx in sample_idx:
    X_sample = X_test_m.iloc[[idx]].values
    if best_model_name in SCALED_MODELS:
        X_sample = scaler_m.transform(X_sample)
    pred = best_model_obj.predict(X_sample)[0]
    prob = best_model_obj.predict_proba(X_sample)[0]
    actual = y_test_m.iloc[idx]
    status = ' Correct' if pred == actual else ' Wrong'
    pred_lbl = 'Pass' if pred==1 else 'Fail'
    actual_lbl = 'Pass' if actual==1 else 'Fail'
    print(f'  Student {idx}: Predicted={pred_lbl} (P={prob[1]:.2f}), Actual={actual_lbl} {status}')

In [ ]:
# 
# FINAL FIGURE  Complete Project Summary
# 
fig = plt.figure(figsize=(18, 10))
fig.suptitle('Student Performance Prediction  Project Summary Dashboard',
             fontsize=15, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# (1) Accuracy bar chart  all models
ax1 = fig.add_subplot(gs[0, :2])
all_models_list = results_mat['Model'].tolist()
acc_mat = results_mat['Accuracy'].tolist()
acc_por = results_por['Accuracy'].tolist()
x = np.arange(len(all_models_list))
ax1.bar(x - 0.2, acc_mat, 0.4, label='Math', color=PALETTE[0], edgecolor='white')
ax1.bar(x + 0.2, acc_por, 0.4, label='Portuguese', color=PALETTE[1], edgecolor='white')
ax1.set_xticks(x)
ax1.set_xticklabels([m.replace(' ','\n') for m in all_models_list], fontsize=8)
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Model Accuracy: Math vs Portuguese')
ax1.legend()
ax1.set_ylim(50, 100)

# (2) G3 distribution
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(mat['G3'], bins=21, color=PALETTE[0], edgecolor='white', alpha=0.8, label='Math')
ax2.hist(por['G3'], bins=21, color=PALETTE[1], edgecolor='white', alpha=0.5, label='Portuguese')
ax2.axvline(10, color='red', linestyle='--', label='Pass threshold')
ax2.set_title('Final Grade Distribution')
ax2.set_xlabel('G3')
ax2.legend(fontsize=8)

# (3) Feature importance - RF
ax3 = fig.add_subplot(gs[1, 0])
rf_row = results_mat[results_mat['Model']=='Random Forest'].iloc[0]
rf_imp = pd.Series(rf_row['Trained Model'].feature_importances_, index=X_mat.columns).nlargest(10)
ax3.barh(rf_imp.index[::-1], rf_imp.values[::-1], color=PALETTE[2], edgecolor='white')
ax3.set_title('Top 10 Features (Random Forest)')
ax3.set_xlabel('Importance')

# (4) Multi-metric comparison
ax4 = fig.add_subplot(gs[1, 1])
metrics_show = ['Accuracy','F1-Score','Precision','Recall']
paper_res = results_mat[results_mat['Model'].isin(paper_avgs)].set_index('Model')
paper_res[metrics_show].plot(kind='bar', ax=ax4, color=PALETTE[:4], edgecolor='white')
ax4.set_title('Multi-Metric (Paper Models, Math)')
ax4.set_ylabel('%')
ax4.set_xticklabels([m.replace(' ','\n') for m in paper_res.index], rotation=15, fontsize=7)
ax4.legend(fontsize=7)
ax4.set_ylim(50, 100)

# (5) CV stability
ax5 = fig.add_subplot(gs[1, 2])
cv_means = results_mat['CV-Accuracy'].tolist()
cv_stds  = results_mat['CV-Std'].tolist()
y_pos = np.arange(len(all_models_list))
ax5.barh(y_pos, cv_means, xerr=cv_stds, color=PALETTE[0], alpha=0.8,
         edgecolor='white', capsize=4)
ax5.set_yticks(y_pos)
ax5.set_yticklabels(all_models_list, fontsize=8)
ax5.set_xlabel('CV Accuracy (%)')
ax5.set_title('5-Fold CV (Math)')
ax5.set_xlim(50, 100)

plt.savefig('fig_summary_dashboard.png', bbox_inches='tight', dpi=120)
plt.show()
print(' Summary dashboard saved.')

In [ ]:
print('\n' + ' '*20)
print('PROJECT COMPLETE  CRISP-DM STUDENT PERFORMANCE PREDICTION')
print(''*60)
print(' Paper: Alsariera et al. (2022)  Computational Intelligence and Neuroscience')
print(' Dataset: UCI Student Performance (Math + Portuguese)')
print(' Models: DT, ANN, SVM, KNN, NB, LogReg + RF, GradBoost (bonus)')
print(''*60)
print(f' Best Paper Model: {best_paper_model["Model"]}  Math: {best_paper_model["Accuracy"]}%')
print(f' Best Overall:     {best_overall_model["Model"]}  Math: {best_overall_model["Accuracy"]}%')
print(''*60)
print(' Output files generated:')
for f in ['fig_grade_distribution.png','fig_categorical_distributions.png',
          'fig_correlation_heatmap.png','fig_feature_relationships.png',
          'fig_grade_progression.png','fig_class_balance.png',
          'fig_accuracy_comparison.png','fig_confusion_matrices.png',
          'fig_feature_importance.png','fig_radar_chart.png',
          'fig_cv_boxplot.png','fig_summary_dashboard.png',
          'best_student_model.pkl','scaler_math.pkl']:
    print(f'    {f}')